# System-Level Energy Optimization

A Rankine power cycle with feedwater preheating via waste heat recovery, modeled entirely with CoolProp real-fluid properties. We use `scipy.optimize.minimize` to find optimal operating points that minimize specific fuel consumption while meeting turbine exhaust quality constraints.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import minimize

from simulations.system_optim.model import system_model_with_recovery

In [ ]:
# Demonstrate the model at a single operating point
result = system_model_with_recovery(
    T_superheat=773.15,  # 500 C
    f_recovery=0.3,
    P_boiler=6e6,
    P_condenser=10e3,
    eta_pump=0.85,
    eta_turbine=0.87,
)
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
def objective(x):
    """Minimize specific fuel consumption."""
    T_sup, f_rec = x
    r = system_model_with_recovery(
        T_superheat=T_sup, f_recovery=f_rec,
    )
    return r["sfc"]


def quality_constraint(x):
    """Turbine exhaust quality must be > 0.85."""
    T_sup, f_rec = x
    r = system_model_with_recovery(
        T_superheat=T_sup, f_recovery=f_rec,
    )
    q = r["exhaust_quality"]
    if q < 0:
        return 1.0  # superheated, no constraint issue
    return q - 0.85


bounds = [(550.0, 900.0), (0.0, 0.8)]

result_opt = minimize(
    objective,
    x0=[750.0, 0.3],
    bounds=bounds,
    constraints={"type": "ineq", "fun": quality_constraint},
    method="SLSQP",
)

print(f"Optimization {'converged' if result_opt.success else 'FAILED'}")
print(f"Optimal T_superheat = {result_opt.x[0]:.1f} K")
print(f"Optimal f_recovery  = {result_opt.x[1]:.3f}")
print(f"Min SFC             = {result_opt.fun:.1f} kJ/kWh")

In [ ]:
# Generate SFC landscape
T_range = np.linspace(550, 900, 40)
f_range = np.linspace(0.0, 0.8, 30)
SFC_grid = np.full((len(f_range), len(T_range)), np.nan)

for i, f_rec in enumerate(f_range):
    for j, T_sup in enumerate(T_range):
        try:
            r = system_model_with_recovery(
                T_superheat=T_sup, f_recovery=f_rec,
            )
            SFC_grid[i, j] = r["sfc"]
        except Exception:
            pass

fig = go.Figure()
fig.add_trace(go.Contour(
    x=T_range - 273.15,
    y=f_range,
    z=SFC_grid,
    colorscale="RdYlGn_r",
    colorbar=dict(title="SFC [kJ/kWh]"),
    contours=dict(showlabels=True),
))
fig.add_trace(go.Scatter(
    x=[result_opt.x[0] - 273.15],
    y=[result_opt.x[1]],
    mode="markers+text",
    marker=dict(size=15, color="red", symbol="star"),
    text=["Optimum"],
    textposition="top center",
    name="Optimal Point",
))
fig.update_layout(
    title="Specific Fuel Consumption Landscape",
    xaxis_title="Superheat Temperature [C]",
    yaxis_title="Heat Recovery Fraction [-]",
    width=800,
    height=600,
)
fig.show()

In [ ]:
# Sensitivity analysis
rows = []
for T_sup in [600, 700, 773.15, 800, 873.15]:
    for f_rec in [0.0, 0.2, 0.4, 0.6]:
        try:
            r = system_model_with_recovery(
                T_superheat=T_sup, f_recovery=f_rec,
            )
            rows.append({
                "T [K]": T_sup,
                "T [C]": round(T_sup - 273.15, 1),
                "f_rec": f_rec,
                "SFC": round(r["sfc"], 1),
                "eta": round(r["thermal_efficiency"], 4),
                "x_exhaust": round(r["exhaust_quality"], 3),
            })
        except Exception:
            pass

df = pd.DataFrame(rows)
df

## Results

The optimizer finds the superheat temperature and heat recovery fraction that minimize specific fuel consumption while keeping turbine exhaust quality above 0.85 (to prevent wet steam erosion).

All thermodynamic properties are computed from CoolProp's IAPWS-IF97 implementation. At zero heat recovery, the model exactly reproduces the standalone Rankine cycle results (verified in the test suite as a self-consistency check).